## BERT4Rec Validation & Marix 추가

In [ ]:
import json, random
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


import wandb

# --- 1) 데이터 로드 & user_seqs 생성 (기존 코드 그대로) ---
with open('./util/result.json','r') as f:
    raw_data = json.load(f)

rows = []
for uid, user in enumerate(raw_data):
    for ts, token in enumerate(user['token_sequence']):
        rows.append([uid, token, ts])
import pandas as pd
df = pd.DataFrame(rows, columns=["user_id","item_id","timestamp"])
user_seqs = df.groupby("user_id")["item_id"].apply(list).tolist()

# --- 2) 토큰 ↔ ID 매핑 (기존 코드 그대로) ---
unique_items = sorted(df["item_id"].unique().tolist())
token2id = {t:i+1 for i,t in enumerate(unique_items)}
token2id['[MASK]'] = len(token2id)+1
id2token = {v:k for k,v in token2id.items()}

# --- 3) Dataset 정의 (val 모드 지원) ---
class BERT4RecDataset(Dataset):
    def __init__(self, sequences, token2id, max_len=20, mask_ratio=0.2, val=False):
        self.sequences     = sequences
        self.token2id      = token2id
        self.max_len       = max_len
        self.mask_ratio    = mask_ratio
        self.val           = val
        self.mask_token_id = token2id['[MASK]']

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        # 1) 토큰→ID, truncate, left-pad
        ids = [self.token2id[t] for t in seq if t in self.token2id]
        ids = ids[-self.max_len:]
        pad = [0]*(self.max_len - len(ids))
        input_ids = pad + ids

        labels     = [-100]*self.max_len
        masked_ids = input_ids.copy()

        if self.val:
            # validation: 마지막 non-pad 토큰만 마스크
            last_pos = max(i for i,x in enumerate(input_ids) if x!=0)
            labels[last_pos]     = input_ids[last_pos]
            masked_ids[last_pos] = self.mask_token_id

        else:
            # train: 랜덤 마스크 + 최소 1개 보장
            for i in range(self.max_len):
                if masked_ids[i]!=0 and random.random()<self.mask_ratio:
                    labels[i]     = masked_ids[i]
                    masked_ids[i] = self.mask_token_id
            if all(l==-100 for l in labels):
                last_pos = max(i for i,x in enumerate(input_ids) if x!=0)
                labels[last_pos]     = input_ids[last_pos]
                masked_ids[last_pos] = self.mask_token_id

        return (
            torch.tensor(masked_ids, dtype=torch.long),
            torch.tensor(labels,     dtype=torch.long),
        )

# --- 4) sequential hold‐out split: train on prefix, val on last ---
#    시퀀스 길이>=2인 것만 사용
valid_seqs = [seq for seq in user_seqs if len(seq)>=2]
train_seqs = [seq[:-1] for seq in valid_seqs]   # [A,B,C]
val_seqs   = valid_seqs                         # [A,B,C,D]

# --- 5) DataLoader 생성 ---
config = {
    "n_layers": 2,
    "n_heads": 2,
    "hidden_size": 64,
    "inner_size": 256,
    "hidden_dropout_prob": 0.2,
    "attn_dropout_prob": 0.2,
    "hidden_act": "gelu",
    "layer_norm_eps": 1e-12,
    "initializer_range": 0.02,
    "mask_ratio": 0.2,
    "loss_type": "CE",
    "max_seq_length": 20,
    "n_items": len(token2id)
}

wandb.init(
    project="seq_rec",                # 원하는 프로젝트 이름
    entity="ai_project_team2",        # 스크린샷에 보인 팀 이름
    name=f"bert4rec_run_{random.randint(1000,9999)}",  # 실험 이름
    config=config
)
train_ds = BERT4RecDataset(train_seqs, token2id,
                           max_len=config['max_seq_length'],
                           mask_ratio=config['mask_ratio'],
                           val=False)
val_ds   = BERT4RecDataset(val_seqs, token2id,
                           max_len=config['max_seq_length'],
                           mask_ratio=0.0,
                           val=True)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False)

# --- 6) 모델·손실·최적화 정의 (기존 코드 그대로) ---
class BERT4Rec(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.hidden_size = cfg['hidden_size']
        self.max_len     = cfg['max_seq_length']
        self.n_items     = cfg['n_items']

        self.item_emb     = nn.Embedding(self.n_items+2, self.hidden_size, padding_idx=0)
        self.pos_emb      = nn.Embedding(self.max_len, self.hidden_size)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.hidden_size, nhead=cfg['n_heads'],
            dim_feedforward=cfg['inner_size'], dropout=cfg['hidden_dropout_prob'],
            activation="gelu", layer_norm_eps=cfg['layer_norm_eps'],
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=cfg['n_layers'])
        self.norm    = nn.LayerNorm(self.hidden_size, eps=cfg['layer_norm_eps'])
        self.drop    = nn.Dropout(cfg['hidden_dropout_prob'])
        self.out     = nn.Linear(self.hidden_size, self.n_items+1)
        self._init_weights(cfg['initializer_range'])

    def _init_weights(self, std):
        for n,p in self.named_parameters():
            if 'weight' in n: nn.init.normal_(p,0,std)
            elif 'bias' in n: nn.init.constant_(p,0)

    def forward(self, input_ids):
        pos = torch.arange(self.max_len, device=input_ids.device) \
                    .unsqueeze(0).expand_as(input_ids)
        x = self.item_emb(input_ids) + self.pos_emb(pos)
        x = self.norm(x); x = self.drop(x)
        pad_mask = (input_ids==0)
        h = self.encoder(x, src_key_padding_mask=pad_mask)
        return self.out(h)

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_bert     = BERT4Rec({**config, **{"n_layers":2,"n_heads":2,"inner_size":256,
                                  "hidden_size":64,"hidden_dropout_prob":0.2,
                                  "layer_norm_eps":1e-12,"initializer_range":0.02,
                                  "n_items":len(token2id)}}).to(device)
opt       = torch.optim.Adam(model_bert.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

# --- 7) 평가 지표 헬퍼 (Precision, Recall, HR, F1 @k) ---
def evaluate_ranking(all_scores, all_labels, ks=[1,5,10]):
    N, V = all_scores.shape
    rank = np.argsort(-all_scores, axis=1)
    metrics = {}
    for k in ks:
        topk = rank[:,:k]
        hits = np.array([1 if all_labels[i] in topk[i] else 0 for i in range(N)])
        prec = hits.mean()/k
        rec  = hits.mean()   # single-ground-truth
        hr   = rec
        f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
        metrics.update({f"P@{k}":prec, f"R@{k}":rec, f"HR@{k}":hr, f"F1@{k}":f1})
    return metrics

# --- 8) Train + Val 루프 ---
for epoch in tqdm(range(1, 31)):
    # -- train --
    model_bert.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        logits = model_bert(x)              # (B, L, V)
        B, L, V = logits.shape
        loss = criterion(logits.view(-1,V), y.view(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    # -- val --
    model_bert.eval()
    val_loss = 0
    all_scores, all_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model_bert(x)
            B, L, V = logits.shape
            val_loss += criterion(logits.view(-1,V), y.view(-1)).item()
            for b in range(B):
                # 1) 마스킹된 위치 (정답이 있는 위치)
                pos = (y[b] != -100).nonzero(as_tuple=True)[0].item()
                scores = logits[b, pos, :].clone()  # (V,)

                # 2) 입력 시퀀스에서 이미 등장한 토큰들은 제외
                input_token_ids = set(x[b].tolist())
                for t_id in input_token_ids:
                    if t_id != 0:  # padding 제외
                        scores[t_id] = float('-inf')

                all_scores.append(scores.cpu().numpy())
                all_labels.append(y[b, pos].item())

    val_loss /= len(val_loader)
    metrics = evaluate_ranking(np.stack(all_scores), np.array(all_labels))

    # print(f"Epoch {epoch:02d}  Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}")
    # for k in [1,5,10]:
    #     print(f"  P@{k}: {metrics[f'P@{k}']:.4f}, R@{k}: {metrics[f'R@{k}']:.4f}, "
    #           f"HR@{k}: {metrics[f'HR@{k}']:.4f}, F1@{k}: {metrics[f'F1@{k}']:.4f}")
    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "P@5": metrics["P@5"],
        "R@5": metrics["R@5"],
        "F1@5": metrics["F1@5"],
        "P@10": metrics["P@10"],
        "R@10": metrics["R@10"],
        "F1@10": metrics["F1@10"],
    })


wandb: Currently logged in as: kwon04210 (listwiserank) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100%|██████████| 30/30 [02:03<00:00,  4.11s/it]


In [4]:
# ── 전제: model 은 학습된 BERT4Rec, token2id/id2token, device 가 정의되어 있다고 가정
max_len       = config['max_seq_length']
mask_token_id = token2id['[MASK]']
# Create 10 random sequences with lengths 8~12
random_seqs = [['CERT_IPE', 'Git', 'AWARD_UNIV', 'CERT_DATA', 'AWARD_OUTER']]
# for _ in range(10):
#     length = random.randint(8, 12)
#     seq = random.choices(unique_items, k=length)
#     random_seqs.append(seq)


model_bert.eval()
with torch.no_grad():
    for idx, seq in enumerate(random_seqs, 1):
        # 1) 토큰 → ID, truncate, left-pad
        ids = [token2id[t] for t in seq]           # Python까지 포함
        ids = ids[-(max_len-1):]                   # 공간 확보를 위해 하나 덜 잘라내고
        masked_ids = ids + [mask_token_id]         # 마지막에 [MASK]를 추가
        pad = [0] * (max_len - len(masked_ids))
        input_ids = pad + masked_ids               # Python 정보가 남아 있게 됨
        # 2) 마지막 non-pad 위치만 MASK
        masked_ids = input_ids.copy()
        masked_ids[max_len-1] = mask_token_id

        # 3) tensor로 변환
        inp = torch.tensor([masked_ids], dtype=torch.long, device=device)
        logits = model_bert(inp)

        # 4) 마스크 위치 logit 추출 후, 중복 제외
        last_logits = logits[0, max_len-1].clone()
        for t_id in set(input_ids):
            if t_id != 0:
                last_logits[t_id] = float('-inf')

        topk_ids    = torch.topk(last_logits, k=5).indices.tolist()
        topk_tokens = [id2token[i] for i in topk_ids]

        print(f"\nTest#{idx}  입력시퀀스: {seq}")
        print(f"추천 Top-5 (중복 제거): {topk_tokens}")




Test#1  입력시퀀스: ['CERT_IPE', 'Git', 'AWARD_UNIV', 'CERT_DATA', 'AWARD_OUTER']
추천 Top-5 (중복 제거): ['TYPE_Maintenance|ROLE_FE|SKILL_HTMLCSS', 'TYPE_Intern|ROLE_AI|SKILL_Python', 'TYPE_Club|ROLE_NUL|SKILL_NUL', 'TYPE_Club|ROLE_FULLSTACK|SKILL_NUL', 'TYPE_Club|ROLE_AI|SKILL_Python']


## BERT4REC Ver2 (Train : Valid : Test = 8 : 1 : 1)

In [1]:
import json, random
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import pandas as pd
import os

import wandb

# ─────────────────────────────────────────────────────────────────────────────
# 재현성을 위한 시드 고정
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# ─────────────────────────────────────────────────────────────────────────────

# 파일 저장 폴더
SAVE_DIR = "./saved_bert4rec"
os.makedirs(SAVE_DIR, exist_ok=True)



# ======================================================
# 1) 데이터 로드 & user_seqs 생성
# ======================================================
with open('./util/result_clean.json','r') as f:
    raw_data = json.load(f)

rows = []
for uid, user in enumerate(raw_data):
    for ts, token in enumerate(user['token_sequence']):
        rows.append([uid, token, ts])
df = pd.DataFrame(rows, columns=["user_id","item_id","timestamp"])
user_seqs = df.groupby("user_id")["item_id"].apply(list).tolist()

# ======================================================
# 2) 토큰 ↔ ID 매핑
# ======================================================
unique_items = sorted(df["item_id"].unique().tolist())
token2id = {t:i+1 for i,t in enumerate(unique_items)}
token2id['[MASK]'] = len(token2id)+1
id2token = {v:k for k,v in token2id.items()}

# token2id를 JSON으로 저장
with open(os.path.join(SAVE_DIR, "token2id.json"), "w", encoding="utf-8") as f:
    json.dump(token2id, f, ensure_ascii=False, indent=2)

# id2token도 저장
id2token = {v: k for k, v in token2id.items()}
with open(os.path.join(SAVE_DIR, "id2token.json"), "w", encoding="utf-8") as f:
    json.dump(id2token, f, ensure_ascii=False, indent=2)

# ======================================================
# 3) 사용자 분할: 80% train / 10% val / 10% test
# ======================================================
random.seed(0)  # 섞기 위한 시드(재현성)
num_users = len(user_seqs)
indices = list(range(num_users))
random.shuffle(indices)

n_train = int(0.8 * num_users)
n_val   = int(0.1 * num_users)
train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train + n_val]
test_idx  = indices[n_train + n_val:]

train_seqs = [user_seqs[i] for i in train_idx]
val_seqs   = [user_seqs[i] for i in val_idx]
test_seqs  = [user_seqs[i] for i in test_idx]

# ======================================================
# 4) Dataset 정의 (BERT4RecDataset)
# ======================================================
class BERT4RecDataset(Dataset):
    def __init__(self, sequences, token2id, max_len=20, mask_ratio=0.2, val=False):
        self.sequences     = sequences
        self.token2id      = token2id
        self.max_len       = max_len
        self.mask_ratio    = mask_ratio
        self.val           = val
        self.mask_token_id = token2id['[MASK]']

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        # 1) 토큰→ID, truncate, left-pad
        ids = [self.token2id[t] for t in seq if t in self.token2id]
        ids = ids[-self.max_len:]
        pad = [0]*(self.max_len - len(ids))
        input_ids = pad + ids

        labels     = [-100]*self.max_len
        masked_ids = input_ids.copy()

        if self.val:
            # validation/Test: 마지막 non-pad 토큰만 마스크 (leave-one-out)
            last_pos = max(i for i,x in enumerate(input_ids) if x!=0)
            labels[last_pos]     = input_ids[last_pos]
            masked_ids[last_pos] = self.mask_token_id
        else:
            # train: 랜덤 마스크 + 최소 1개 보장
            for i in range(self.max_len):
                if masked_ids[i]!=0 and random.random() < self.mask_ratio:
                    labels[i]       = masked_ids[i]
                    masked_ids[i]   = self.mask_token_id
            if all(l == -100 for l in labels):
                last_pos = max(i for i,x in enumerate(input_ids) if x!=0)
                labels[last_pos]     = input_ids[last_pos]
                masked_ids[last_pos] = self.mask_token_id

        return (
            torch.tensor(masked_ids, dtype=torch.long),
            torch.tensor(labels,     dtype=torch.long),
        )

# ======================================================
# 5) DataLoader 생성
# ======================================================
config = {
    "n_layers": 4,
    "n_heads": 4,
    "hidden_size": 64,
    "inner_size": 256,
    "hidden_dropout_prob": 0.2,
    "attn_dropout_prob": 0.2,
    "hidden_act": "gelu",
    "layer_norm_eps": 1e-12,
    "initializer_range": 0.02,
    "mask_ratio": 0.2,
    "loss_type": "CE",
    "max_seq_length": 20,
    "n_items": len(token2id)
}

wandb.init(
    project="seq_rec",                # 원하는 프로젝트 이름
    entity="ai_project_team2",        # 스크린샷에 보인 팀 이름
    name=f"bert4rec_run_test_4_4",  # 실험 이름
    config=config
)

BATCH_SIZE = 128

# Train DataLoader
train_ds = BERT4RecDataset(
    train_seqs,
    token2id,
    max_len=config['max_seq_length'],
    mask_ratio=config['mask_ratio'],
    val=False
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

# Validation DataLoader
val_ds = BERT4RecDataset(
    val_seqs,
    token2id,
    max_len=config['max_seq_length'],
    mask_ratio=0.0,
    val=True
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# Test DataLoader
test_ds = BERT4RecDataset(
    test_seqs,
    token2id,
    max_len=config['max_seq_length'],
    mask_ratio=0.0,
    val=True
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# ======================================================
# 6) BERT4Rec 모델 정의
# ======================================================
class BERT4Rec(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.hidden_size = cfg['hidden_size']
        self.max_len     = cfg['max_seq_length']
        self.n_items     = cfg['n_items']

        # +2: padding 0, mask 토큰
        self.item_emb = nn.Embedding(self.n_items+2, self.hidden_size, padding_idx=0)
        self.pos_emb  = nn.Embedding(self.max_len, self.hidden_size)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.hidden_size, nhead=cfg['n_heads'],
            dim_feedforward=cfg['inner_size'], dropout=cfg['hidden_dropout_prob'],
            activation="gelu", layer_norm_eps=cfg['layer_norm_eps'],
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=cfg['n_layers'])
        self.norm    = nn.LayerNorm(self.hidden_size, eps=cfg['layer_norm_eps'])
        self.drop    = nn.Dropout(cfg['hidden_dropout_prob'])
        self.out     = nn.Linear(self.hidden_size, self.n_items+1)
        self._init_weights(cfg['initializer_range'])

    def _init_weights(self, std):
        for n,p in self.named_parameters():
            if 'weight' in n:
                nn.init.normal_(p, 0, std)
            elif 'bias' in n:
                nn.init.constant_(p, 0)

    def forward(self, input_ids):
        pos = torch.arange(self.max_len, device=input_ids.device).unsqueeze(0).expand_as(input_ids)
        x = self.item_emb(input_ids) + self.pos_emb(pos)
        x = self.norm(x)
        x = self.drop(x)

        pad_mask = (input_ids == 0)
        h = self.encoder(x, src_key_padding_mask=pad_mask)
        return self.out(h)

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_bert = BERT4Rec({**config, 
                       **{"n_layers":4, "n_heads":4, "inner_size":256,
                          "hidden_size":64, "hidden_dropout_prob":0.2,
                          "layer_norm_eps":1e-12, "initializer_range":0.02,
                          "n_items":len(token2id)}}).to(device)

opt       = torch.optim.Adam(model_bert.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

# ======================================================
# 7) Validation에서 사용하는 기존 evaluate_ranking 함수 (변경 없음)
# ======================================================
def evaluate_ranking(all_scores, all_labels, ks=[1, 5, 10]):
    """
    기존 평가용: P@k, R@k, HR@k, F1@k, nDCG@k를 계산한다.
    """
    N, V = all_scores.shape
    rank = np.argsort(-all_scores, axis=1)
    metrics = {}
    for k in ks:
        topk_indices = rank[:, :k]
        hits = np.array([1 if all_labels[i] in topk_indices[i] else 0 for i in range(N)])
        prec = hits.mean() / k
        rec  = hits.mean()
        hr   = rec
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

        dcg_list = []
        for i in range(N):
            topk_full = rank[i][:k]
            if all_labels[i] in topk_full:
                r = int(np.where(topk_full == all_labels[i])[0][0])
                dcg_i = 1.0 / np.log2(r + 2)
            else:
                dcg_i = 0.0
            dcg_list.append(dcg_i)

        ndcg = float(np.mean(dcg_list))
        metrics.update({
            f"P@{k}":   prec,
            f"R@{k}":   rec,
            f"HR@{k}":  hr,
            f"F1@{k}":  f1,
            f"nDCG@{k}": ndcg,
        })
    return metrics

# ======================================================
# 8) Train + Validation 루프: Validation 손실이 가장 낮은 모델 저장
# ======================================================
best_val_loss = float('inf')
best_epoch    = -1
EPOCHS = 1000

for epoch in tqdm(range(1, 1 + EPOCHS)):
    # --- (1) Train ---
    model_bert.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        logits = model_bert(x)              # (B, L, V)
        B, L, V = logits.shape
        loss = criterion(logits.view(-1, V), y.view(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    # --- (2) Validation ---
    model_bert.eval()
    val_loss = 0
    all_scores, all_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model_bert(x)
            B, L, V = logits.shape
            val_loss += criterion(logits.view(-1, V), y.view(-1)).item()

            for b in range(B):
                pos = (y[b] != -100).nonzero(as_tuple=True)[0].item()
                scores = logits[b, pos, :].clone()

                input_token_ids = set(x[b].tolist())
                for t_id in input_token_ids:
                    if t_id != 0:
                        scores[t_id] = float('-inf')

                all_scores.append(scores.cpu().numpy())
                all_labels.append(y[b, pos].item())

    val_loss /= len(val_loader)
    val_metrics = evaluate_ranking(np.stack(all_scores), np.array(all_labels))

    # --- (3) 가장 낮은 Validation 손실 시 모델 저장 ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch    = epoch
        save_path = os.path.join(SAVE_DIR, "best_bert4rec.pt")
        torch.save(model_bert.state_dict(), save_path)

    # --- (4) (선택) Validation 결과 출력 ---
    # print(f"Epoch {epoch:02d}  Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}")
    # print(f"  P@5: {val_metrics['P@5']:.4f}, R@5: {val_metrics['R@5']:.4f}, F1@5: {val_metrics['F1@5']:.4f}")
    wandb.log({
    "epoch": epoch,
    "train_loss": train_loss,
    "val_loss": val_loss,
    })

print(f"▶ Stored Model  : Epoch {best_epoch}  |  Val Loss {best_val_loss:.4f}")

# ======================================================
# 9) Test 평가: HR@1, HR@5, HR@10, NDCG@5, NDCG@10, MRR 계산
# ======================================================

def evaluate_simple_metrics(all_scores: np.ndarray, all_labels: np.ndarray):

    N, V = all_scores.shape
    # 1) 점수를 내림차순 정렬한 인덱스 행렬
    rank = np.argsort(-all_scores, axis=1)  # shape=(N, V)

    # --- HR@1, HR@5, HR@10 계산 ---
    hits1  = np.array([1 if all_labels[i] in rank[i, :1] else 0 for i in range(N)])
    hits5  = np.array([1 if all_labels[i] in rank[i, :5] else 0 for i in range(N)])
    hits10 = np.array([1 if all_labels[i] in rank[i, :10] else 0 for i in range(N)])
    HR1  = hits1.mean()
    HR5  = hits5.mean()
    HR10 = hits10.mean()

    # --- NDCG@5, NDCG@10 계산 (단일 정답이므로 IDCG=1 고정) ---
    def compute_ndcg_at_k(k):
        dcg_list = np.zeros(N, dtype=np.float32)
        for i in range(N):
            true_item = all_labels[i]
            topk_items = rank[i, :k]
            if true_item in topk_items:
                r = int(np.where(topk_items == true_item)[0][0])  # 0-based index
                dcg_list[i] = 1.0 / np.log2(r + 2.0)
            else:
                dcg_list[i] = 0.0
        return float(dcg_list.mean())

    NDCG5  = compute_ndcg_at_k(5)
    NDCG10 = compute_ndcg_at_k(10)

    # --- MRR 계산 ---
    rr_list = np.zeros(N, dtype=np.float32)
    for i in range(N):
        true_item = all_labels[i]
        r_full = int(np.where(rank[i] == true_item)[0][0])  # 전체 V크기 랭킹에서의 위치
        rr_list[i] = 1.0 / (r_full + 1.0)
    MRR = float(rr_list.mean())

    return {
        "HR@1":  HR1,
        "HR@5":  HR5,
        "HR@10": HR10,
        "NDCG@5":  NDCG5,
        "NDCG@10": NDCG10,
        "MRR":   MRR
    }

# --- (1) 저장된 최적 모델 로드 ---
model_bert.load_state_dict(torch.load(save_path))
model_bert.eval()

# --- (2) Test 데이터에서 점수와 정답 수집 ---
test_loss = 0
all_scores, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        logits = model_bert(x)           # (B, L, V)
        B, L, V = logits.shape
        test_loss += criterion(logits.view(-1, V), y.view(-1)).item()

        for b in range(B):
            pos = (y[b] != -100).nonzero(as_tuple=True)[0].item()
            scores = logits[b, pos, :].clone()  # (V,)

            # 이미 시퀀스에 등장한 토큰 제외
            input_token_ids = set(x[b].tolist())
            for t_id in input_token_ids:
                if t_id != 0:
                    scores[t_id] = float('-inf')

            all_scores.append(scores.cpu().numpy())
            all_labels.append(y[b, pos].item())

test_loss /= len(test_loader)

# --- (3) Test 지표 계산 ---
all_scores_np = np.stack(all_scores)           # shape=(N_test, V)
all_labels_np = np.array(all_labels, dtype=int) # shape=(N_test,)

test_metrics = evaluate_simple_metrics(all_scores_np, all_labels_np)

# --- (4) Test 결과 출력 ---
print("\n===== Test 결과 =====")
print(f"Test Loss: {test_loss:.4f}")
print(f"HR@1:   {test_metrics['HR@1']:.4f}")
print(f"HR@5:   {test_metrics['HR@5']:.4f}")
print(f"HR@10:  {test_metrics['HR@10']:.4f}")
print(f"NDCG@5: {test_metrics['NDCG@5']:.4f}")
print(f"NDCG@10:{test_metrics['NDCG@10']:.4f}")
print(f"MRR:    {test_metrics['MRR']:.4f}")


wandb: Currently logged in as: kwon04210 (listwiserank) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100%|██████████| 1000/1000 [20:28<00:00,  1.23s/it]


▶ Stored Model  : Epoch 782  |  Val Loss 2.2578

===== Test 결과 =====
Test Loss: 2.3842
HR@1:   0.3577
HR@5:   0.6988
HR@10:  0.8053
NDCG@5: 0.5371
NDCG@10:0.5716
MRR:    0.5060


In [20]:
import json, random
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import pandas as pd
import os

import wandb

# ─────────────────────────────────────────────────────────────────────────────
# 재현성을 위한 시드 고정
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# ─────────────────────────────────────────────────────────────────────────────

# 파일 저장 폴더
SAVE_DIR = "./saved_bert4rec"
os.makedirs(SAVE_DIR, exist_ok=True)

# ======================================================
# 1) 데이터 로드 & user_seqs 생성 (변경 없음)
# ======================================================
with open('./util/result_clean.json','r') as f:
    raw_data = json.load(f)

rows = []
for uid, user in enumerate(raw_data):
    for ts, token in enumerate(user['token_sequence']):
        rows.append([uid, token, ts])
df = pd.DataFrame(rows, columns=["user_id","item_id","timestamp"])
user_seqs = df.groupby("user_id")["item_id"].apply(list).tolist()

# ======================================================
# 2) 토큰 ↔ ID 매핑 (변경 없음)
# ======================================================
unique_items = sorted(df["item_id"].unique().tolist())
token2id = {t:i+1 for i,t in enumerate(unique_items)}
token2id['[MASK]'] = len(token2id) + 1
id2token = {v:k for k,v in token2id.items()}

# JSON으로 저장
with open(os.path.join(SAVE_DIR, "token2id.json"), "w", encoding="utf-8") as f:
    json.dump(token2id, f, ensure_ascii=False, indent=2)
with open(os.path.join(SAVE_DIR, "id2token.json"), "w", encoding="utf-8") as f:
    json.dump(id2token, f, ensure_ascii=False, indent=2)

# ======================================================
# 3) Leave-two-out 방식으로 분할:
#      - train_seqs: seq[:-2]
#      - val_seqs:   seq[:-1]
#      - test_seqs:  seq (원본 전체, 마지막 아이템이 테스트 대상)
# ======================================================
train_seqs = []
val_seqs   = []
test_seqs  = []

for seq in user_seqs:
    # 길이가 2 이하인 경우는 skip (train/val/test가 모두 성립하려면 최소 3개 이상)
    if len(seq) < 3:
        continue

    # train: 마지막 두 개 아이템 제외
    train_seqs.append(seq[:-2])

    # validation: 마지막 아이템 한 개만 제외 → BERT4RecDataset(val=True)에서
    #             마지막 non-pad 위치(=원본 두 번째 마지막)를 마스킹하여 평가
    val_seqs.append(seq[:-1])

    # test: 원본 전체 시퀀스 → BERT4RecDataset(val=True)에서
    #       마지막 non-pad 위치(=원본 마지막)를 마스킹하여 평가
    test_seqs.append(seq)

# ======================================================
# 4) Dataset & DataLoader 정의
# ======================================================
class BERT4RecDataset(Dataset):
    def __init__(self, sequences, token2id, max_len=20, mask_ratio=0.2, val=False):
        self.sequences     = sequences
        self.token2id      = token2id
        self.max_len       = max_len
        self.mask_ratio    = mask_ratio
        self.val           = val
        self.mask_token_id = token2id['[MASK]']

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        # 1) 토큰→ID, truncate, left-pad
        ids = [self.token2id[t] for t in seq if t in self.token2id]
        ids = ids[-self.max_len:]
        pad = [0] * (self.max_len - len(ids))
        input_ids = pad + ids

        labels     = [-100] * self.max_len
        masked_ids = input_ids.copy()

        if self.val:
            # validation/Test: 마지막 non-pad 토큰만 마스크
            last_pos = max(i for i, x in enumerate(input_ids) if x != 0)
            labels[last_pos]     = input_ids[last_pos]
            masked_ids[last_pos] = self.mask_token_id
        else:
            # train: 랜덤 마스크 + 최소 1개 보장
            for i in range(self.max_len):
                if masked_ids[i] != 0 and random.random() < self.mask_ratio:
                    labels[i]       = masked_ids[i]
                    masked_ids[i]   = self.mask_token_id
            if all(l == -100 for l in labels):
                # 만약 모든 위치에서 마스킹이 일어나지 않았다면, 거듭 확인하여 
                # 마지막 non-pad 위치는 무조건 한 번 마스킹 처리
                last_pos = max(i for i,x in enumerate(input_ids) if x != 0)
                labels[last_pos]     = input_ids[last_pos]
                masked_ids[last_pos] = self.mask_token_id

        return (
            torch.tensor(masked_ids, dtype=torch.long),
            torch.tensor(labels,     dtype=torch.long),
        )

config = {
    "n_layers": 4,
    "n_heads": 4,
    "hidden_size": 64,
    "inner_size": 256,
    "hidden_dropout_prob": 0.2,
    "attn_dropout_prob": 0.2,
    "hidden_act": "gelu",
    "layer_norm_eps": 1e-12,
    "initializer_range": 0.02,
    "mask_ratio": 0.2,
    "loss_type": "CE",
    "max_seq_length": 20,
    "n_items": len(token2id)
}

wandb.init(
    project="seq_rec",
    entity="ai_project_team2",
    name="bert4rec_leave2out",
    config=config
)

BATCH_SIZE = 128

# ─ Train DataLoader ─
train_ds = BERT4RecDataset(
    train_seqs,
    token2id,
    max_len=config['max_seq_length'],
    mask_ratio=config['mask_ratio'],  # train 단계: 랜덤 마스크
    val=False
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

# ─ Validation DataLoader ─
val_ds = BERT4RecDataset(
    val_seqs,
    token2id,
    max_len=config['max_seq_length'],
    mask_ratio=0.0,  # val 단계: mask_ratio=0.0으로 설정
    val=True         # val=True → 마지막 non-pad 위치(=원본 seq의 penultimate)를 마스킹
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# ─ Test DataLoader ─
test_ds = BERT4RecDataset(
    test_seqs,
    token2id,
    max_len=config['max_seq_length'],
    mask_ratio=0.0,  # test 단계: mask_ratio=0.0
    val=True         # val=True → 마지막 non-pad 위치(=원본 seq의 마지막)를 마스킹
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# ======================================================
# 5) BERT4Rec 모델 정의 (이전과 동일)
# ======================================================
class BERT4Rec(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.hidden_size = cfg['hidden_size']
        self.max_len     = cfg['max_seq_length']
        self.n_items     = cfg['n_items']

        # +2: padding(0), mask 토큰
        self.item_emb = nn.Embedding(self.n_items+2, self.hidden_size, padding_idx=0)
        self.pos_emb  = nn.Embedding(self.max_len, self.hidden_size)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.hidden_size, nhead=cfg['n_heads'],
            dim_feedforward=cfg['inner_size'], dropout=cfg['hidden_dropout_prob'],
            activation="gelu", layer_norm_eps=cfg['layer_norm_eps'],
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=cfg['n_layers'])
        self.norm    = nn.LayerNorm(self.hidden_size, eps=cfg['layer_norm_eps'])
        self.drop    = nn.Dropout(cfg['hidden_dropout_prob'])
        self.out     = nn.Linear(self.hidden_size, self.n_items+1)
        self._init_weights(cfg['initializer_range'])

    def _init_weights(self, std):
        for n,p in self.named_parameters():
            if 'weight' in n:
                nn.init.normal_(p, 0, std)
            elif 'bias' in n:
                nn.init.constant_(p, 0)

    def forward(self, input_ids):
        pos = torch.arange(self.max_len, device=input_ids.device).unsqueeze(0).expand_as(input_ids)
        x = self.item_emb(input_ids) + self.pos_emb(pos)
        x = self.norm(x)
        x = self.drop(x)

        pad_mask = (input_ids == 0)
        h = self.encoder(x, src_key_padding_mask=pad_mask)
        return self.out(h)

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_bert = BERT4Rec({**config, **{"n_items": len(token2id)}}).to(device)

opt       = torch.optim.Adam(model_bert.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

# ======================================================
# 6) Train + Validation 루프
#    - Validation Loss가 가장 낮을 때 모델 저장
# ======================================================
best_val_loss = float('inf')
best_epoch    = -1
EPOCHS = 100

for epoch in tqdm(range(1, 1 + EPOCHS)):
    # --- (1) Train ---
    model_bert.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        logits = model_bert(x)              # (B, L, V)
        B, L, V = logits.shape
        loss = criterion(logits.view(-1, V), y.view(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    # --- (2) Validation ---
    model_bert.eval()
    val_loss = 0
    all_scores, all_labels = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model_bert(x)           # (B, L, V)
            B, L, V = logits.shape
            val_loss += criterion(logits.view(-1, V), y.view(-1)).item()

            # 평가 시: penultimate (=val_targets)만 예측
            for b in range(B):
                pos = (y[b] != -100).nonzero(as_tuple=True)[0].item()  # 마지막 non-pad 위치
                scores = logits[b, pos, :].clone()

                # 이미 시퀀스에 등장한 아이템 제외
                input_token_ids = set(x[b].tolist())
                for t_id in input_token_ids:
                    if t_id != 0:
                        scores[t_id] = float('-inf')

                all_scores.append(scores.cpu().numpy())
                all_labels.append(y[b, pos].item())

    val_loss /= len(val_loader)
    # P@5, R@5, HR@5, F1@5, nDCG@5 등 계산:
    val_metrics = {}
    # (이하 val_metrics 계산 함수 호출 부분은 앞서 정의된 evaluate_ranking 이용)
    val_metrics = evaluate_ranking(np.stack(all_scores), np.array(all_labels))

    # --- (3) Validation Loss 기준으로 모델 저장 ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch    = epoch
        save_path = os.path.join(SAVE_DIR, "best_bert4rec_leave2out.pt")
        torch.save(model_bert.state_dict(), save_path)

    # --- (4) wandb 로깅/출력 (필요 시) ---
    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_P@5":  val_metrics['P@5'],
        "val_R@5":  val_metrics['R@5'],
        "val_HR@5": val_metrics['HR@5'],
        "val_F1@5": val_metrics['F1@5'],
        "val_nDCG@5": val_metrics['nDCG@5'],
    })

print(f"▶ Best Model (Leave-2-Out) 저장 → Epoch {best_epoch} | Val Loss {best_val_loss:.4f}")

# ======================================================
# 7) Test 평가 (Leave-two-out)
#    - 저장된 best 모델 불러와서 penultimate가 아니라 “마지막” 토큰 평가
# ======================================================
def evaluate_simple_metrics(all_scores: np.ndarray, all_labels: np.ndarray):
    N, V = all_scores.shape
    rank = np.argsort(-all_scores, axis=1)

    hits1  = np.array([1 if all_labels[i] in rank[i, :1] else 0 for i in range(N)])
    hits5  = np.array([1 if all_labels[i] in rank[i, :5] else 0 for i in range(N)])
    hits10 = np.array([1 if all_labels[i] in rank[i, :10] else 0 for i in range(N)])
    HR1  = hits1.mean()
    HR5  = hits5.mean()
    HR10 = hits10.mean()

    def compute_ndcg_at_k(k):
        dcg_list = np.zeros(N, dtype=np.float32)
        for i in range(N):
            true_item = all_labels[i]
            topk_items = rank[i, :k]
            if true_item in topk_items:
                r = int(np.where(topk_items == true_item)[0][0])
                dcg_list[i] = 1.0 / np.log2(r + 2.0)
            else:
                dcg_list[i] = 0.0
        return float(dcg_list.mean())

    NDCG5  = compute_ndcg_at_k(5)
    NDCG10 = compute_ndcg_at_k(10)

    rr_list = np.zeros(N, dtype=np.float32)
    for i in range(N):
        true_item = all_labels[i]
        r_full = int(np.where(rank[i] == true_item)[0][0])
        rr_list[i] = 1.0 / (r_full + 1.0)
    MRR = float(rr_list.mean())

    return {
        "HR@1":   HR1,
        "HR@5":   HR5,
        "HR@10":  HR10,
        "NDCG@5":  NDCG5,
        "NDCG@10": NDCG10,
        "MRR":    MRR
    }

# (1) 저장된 최적 모델 불러오기
model_bert.load_state_dict(torch.load(save_path))
model_bert.eval()

# (2) Test 데이터(leave-two-out의 “원본 전체 시퀀스”)에서 점수와 정답 수집
test_loss = 0
all_scores, all_labels = [], []

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        logits = model_bert(x)           # (B, L, V)
        B, L, V = logits.shape
        test_loss += criterion(logits.view(-1, V), y.view(-1)).item()

        for b in range(B):
            # y[b]에서 -100이 아닌 위치 = “마지막 non-pad 위치” (원본 시퀀스의 마지막 아이템)
            pos = (y[b] != -100).nonzero(as_tuple=True)[0].item()
            scores = logits[b, pos, :].clone()  # (V,)

            # 이미 시퀀스에 등장한 아이템(excluding pad=0)은 점수 -inf 처리
            input_token_ids = set(x[b].tolist())
            for t_id in input_token_ids:
                if t_id != 0:
                    scores[t_id] = float('-inf')

            all_scores.append(scores.cpu().numpy())
            all_labels.append(y[b, pos].item())

test_loss /= len(test_loader)

# (3) Test 지표 계산
all_scores_np = np.stack(all_scores)
all_labels_np = np.array(all_labels, dtype=int)
test_metrics = evaluate_simple_metrics(all_scores_np, all_labels_np)

# (4) Test 결과 출력
print("\n===== Leave-2-Out Test 결과 =====")
print(f"Test Loss: {test_loss:.4f}")
print(f"HR@1:   {test_metrics['HR@1']:.4f}")
print(f"HR@5:   {test_metrics['HR@5']:.4f}")
print(f"HR@10:  {test_metrics['HR@10']:.4f}")
print(f"NDCG@5: {test_metrics['NDCG@5']:.4f}")
print(f"NDCG@10:{test_metrics['NDCG@10']:.4f}")
print(f"MRR:    {test_metrics['MRR']:.4f}")


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▁▁▂▂▂▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train_loss,█▆▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▂▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,███▇▇▅▄▄▃▃▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,1000
train_loss,1.95191
val_loss,2.30465


100%|██████████| 100/100 [04:52<00:00,  2.93s/it]


▶ Best Model (Leave-2-Out) 저장 → Epoch 100 | Val Loss 3.1424

===== Leave-2-Out Test 결과 =====
Test Loss: 3.3357
HR@1:   0.2024
HR@5:   0.4911
HR@10:  0.6442
NDCG@5: 0.3503
NDCG@10:0.3995
MRR:    0.3407


In [2]:
import random
import torch

# ─────────────────────────────────────────────────────────────────────────────
# (가정) 이미 아래 변수들이 정의되어 있다고 가정합니다:
#   model_bert: 학습된 BERT4Rec 모델 (torch.nn.Module)
#   token2id:   {아이템이름: ID, ..., "[MASK]": some_id}
#   id2token:   {ID: 아이템이름, ...}
#   device:     "cuda" 또는 "cpu"
#   unique_items: 전체 아이템 이름 리스트 (예: df["item_id"].unique().tolist())
#   config["max_seq_length"]: 최대 시퀀스 길이 (예: 20)
# ─────────────────────────────────────────────────────────────────────────────

max_len       = config['max_seq_length']
mask_token_id = token2id['[MASK]']

# 예시 1개 시퀀스만 직접 지정 (길이가 5)
random_seqs = [
    ['Python', 'PyTorch']
]

model_bert.eval()

with torch.no_grad():
    for idx, seq in enumerate(random_seqs, 1):
        # 1) 토큰 → ID 변환 (토큰이 매핑에 없으면 에러가 나므로, 반드시 token2id 내에 존재하는 토큰만 사용)
        ids = [token2id[t] for t in seq]

        # 2) max_len-1 만큼만 뒤쪽을 남기기 (마지막에 [MASK] 자리를 하나 확보)
        ids = ids[-(max_len - 1):]  # e.g. max_len=20이면 max_len-1=19개까지만 남김

        # 3) 마지막에 [MASK] 토큰 추가
        masked_ids = ids + [mask_token_id]  # 길이가 <= max_len

        # 4) 남는 부분은 0 (padding)으로 채우기
        pad = [0] * (max_len - len(masked_ids))
        input_ids = pad + masked_ids      # 길이 = max_len

        # 5) 텐서로 변환 후 device로 이동
        inp = torch.tensor([input_ids], dtype=torch.long, device=device)  # shape=(1, max_len)

        # 6) 모델에 넣어서 로짓 계산
        logits = model_bert(inp)  # shape=(1, max_len, V) , V = n_items+1 (padding 제외)

        # 7) 마스크된 위치의 로짓 벡터만 추출 (맨 마지막 인덱스 = max_len-1)
        last_logits = logits[0, max_len - 1].clone()  # shape=(V,)

        # 8) 이미 시퀀스에 포함된 토큰 ID들은 후보에서 제외 (score=-inf)
        #    input_ids에는 padding(0), 실제 아이템 ID, [MASK] ID 등이 들어 있음
        for t_id in set(input_ids):
            if t_id != 0:           # padding(0)은 제외
                last_logits[t_id] = float('-inf')

        # 9) top-5 후보 ID 추출
        topk_ids = torch.topk(last_logits, k=5).indices.tolist()

        # 10) ID → 실제 토큰(아이템 이름)으로 변환
        topk_tokens = [id2token[i] for i in topk_ids]

        # 결과 출력
        print(f"\nTest#{idx}  입력시퀀스: {seq}")
        print(f"추천 Top-5 (중복 제외): {topk_tokens}")


KeyError: '[MASK]'

In [2]:
token2id

{'AWARD_OUTER': 1,
 'AWARD_UNIV': 2,
 'AWS': 3,
 'Angular': 4,
 'Ansible': 5,
 'ApacheSpark': 6,
 'Axios': 7,
 'Azure': 8,
 'AzureDevOps': 9,
 'Bash': 10,
 'Bootstrap': 11,
 'CERT_AI': 12,
 'CERT_AWS': 13,
 'CERT_DATA': 14,
 'CERT_IPE': 15,
 'CERT_OUTER': 16,
 'CUDA': 17,
 'Celery': 18,
 'CircleCI': 19,
 'Cypress': 20,
 'Django': 21,
 'Docker': 22,
 'DockerCompose': 23,
 'DynamoDB': 24,
 'ELK': 25,
 'Electron': 26,
 'Expo': 27,
 'Express': 28,
 'FAISS': 29,
 'FastAPI': 30,
 'Flask': 31,
 'Fluentd': 32,
 'GCP': 33,
 'Gin': 34,
 'Git': 35,
 'GitLabCI': 36,
 'Go': 37,
 'Grafana': 38,
 'HTMLCSS': 39,
 'Helm': 40,
 'JS': 41,
 'JWT': 42,
 'Java': 43,
 'JavaScript': 44,
 'Jenkins': 45,
 'Jest': 46,
 'Kafka': 47,
 'Keras': 48,
 'Kotlin': 49,
 'Kubernetes': 50,
 'Matplotlib': 51,
 'Milvus': 52,
 'MongoDB': 53,
 'MySQL': 54,
 'Next': 55,
 'NgRx': 56,
 'Node': 57,
 'NumPy': 58,
 'Numpy': 59,
 'Nuxt': 60,
 'OAuth2': 61,
 'Oracle': 62,
 'PHP': 63,
 'Pandas': 64,
 'PostgreSQL': 65,
 'PowerShell': 66

## SASRec Validation & Matrix

In [ ]:
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import wandb
import pandas as pd

# ==================== 1. Load & preprocess ====================
with open('../util/result.json','r') as f:
    raw = json.load(f)
rows = []
for uid, user in enumerate(raw):
    for ts, tok in enumerate(user['token_sequence']):
        rows.append((uid, tok, ts))
_df = pd.DataFrame(rows, columns=['user_id','item_id','timestamp'])
user_seqs = _df.groupby('user_id')['item_id'].apply(list).tolist()
valid_seqs = [seq for seq in user_seqs if len(seq) >= 2]
unique_items = sorted(_df['item_id'].unique())
token2id = {t: i+1 for i, t in enumerate(unique_items)}  # 0 = PAD
id2token = {i: t for t, i in token2id.items()}

# ==================== 2. Dataset definitions ====================
class SeqLabelDataset(Dataset):
    """
    Full-sequence dataset for predicting next item at each position
    """
    def __init__(self, sequences, t2i, max_len=20):
        self.seqs    = sequences
        self.t2i     = t2i
        self.max_len = max_len
    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, idx):
        ids = [self.t2i[t] for t in self.seqs[idx] if t in self.t2i]
        ids = ids[-self.max_len:]
        L = len(ids)
        pad_len   = self.max_len - L
        input_ids = [0]*pad_len + ids
        labels    = [0]*pad_len + ids[1:] + [0]
        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(labels,    dtype=torch.long)
        )

class SASRecDataset(Dataset):
    """
    Hold-out last item for recommendation metrics
    """
    def __init__(self, sequences, t2i, max_len=20):
        self.seqs    = sequences
        self.t2i     = t2i
        self.max_len = max_len
    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, idx):
        ids = [self.t2i[t] for t in self.seqs[idx] if t in self.t2i]
        ids = ids[-self.max_len:]
        prefix, target = ids[:-1], ids[-1]
        L = len(prefix)
        pad_len = self.max_len - L
        input_ids = [0]*pad_len + prefix
        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(L,         dtype=torch.long),
            torch.tensor(target,    dtype=torch.long)
        )

# ==================== 3. SASRec model ====================
class SASRec(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.max_len     = cfg['max_seq_length']
        self.hidden_size = cfg['hidden_size']
        self.n_items     = cfg['n_items']
        self.item_emb = nn.Embedding(self.n_items+1, self.hidden_size, padding_idx=0)
        self.pos_emb  = nn.Embedding(self.max_len, self.hidden_size)
        enc = nn.TransformerEncoderLayer(
            d_model=self.hidden_size,
            nhead=cfg['n_heads'],
            dim_feedforward=cfg['inner_size'],
            dropout=cfg['hidden_dropout_prob'],
            activation=cfg['hidden_act'],
            layer_norm_eps=cfg['layer_norm_eps'],
            batch_first=True
        )
        self.encoder     = nn.TransformerEncoder(enc, num_layers=cfg['n_layers'])
        self.layer_norm  = nn.LayerNorm(self.hidden_size, eps=cfg['layer_norm_eps'])
        self.dropout     = nn.Dropout(cfg['hidden_dropout_prob'])
        self.output_bias = nn.Parameter(torch.zeros(self.n_items+1))
        self._init_weights(cfg['initializer_range'])
    def _init_weights(self, std):
        for n, p in self.named_parameters():
            if 'weight' in n:
                nn.init.normal_(p, mean=0.0, std=std)
            elif 'bias' in n:
                nn.init.constant_(p, 0.0)
    def forward(self, input_ids):
        B, L = input_ids.size()
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)
        x = self.item_emb(input_ids) + self.pos_emb(pos)
        x = self.layer_norm(x)
        x = self.dropout(x)
        mask = torch.triu(torch.ones((L, L), device=x.device), diagonal=1).bool()
        h = self.encoder(x, mask=mask)
        logits = torch.matmul(h, self.item_emb.weight.t()) + self.output_bias  # (B, L, V)
        return logits

# ==================== 4. Config & W&B ====================
config = {
    'n_layers':2, 'n_heads':2, 'hidden_size':64,
    'inner_size':256, 'hidden_dropout_prob':0.5,
    'hidden_act':'gelu', 'layer_norm_eps':1e-12,
    'initializer_range':0.02,
    'max_seq_length':20,
    'n_items': len(token2id)
}
wandb.init(
    project='seq_rec', entity='ai_project_team2',
    name=f'sasrec_run_{random.randint(1000,9999)}', config=config
)

# ==================== 5. DataLoaders ====================
train_ds      = SeqLabelDataset(valid_seqs, token2id, max_len=config['max_seq_length'])
val_rec_ds    = SASRecDataset(valid_seqs, token2id, max_len=config['max_seq_length'])
val_seq_ds    = SeqLabelDataset(valid_seqs, token2id, max_len=config['max_seq_length'])
train_loader  = DataLoader(train_ds, batch_size=4, shuffle=True,  drop_last=True)
val_rec_loader= DataLoader(val_rec_ds, batch_size=4, shuffle=False)
val_seq_loader= DataLoader(val_seq_ds, batch_size=4, shuffle=False)

# ==================== 6. Loss, model, optimizer ====================
device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_sas        = SASRec(config).to(device)
criterion_ce = nn.CrossEntropyLoss(ignore_index=0)
optimizer    = torch.optim.Adam(model_sas.parameters(), lr=1e-3)

# ==================== 7. Train & Validation ====================
for epoch in range(1, 51):
    model_sas.train()
    train_loss = 0
    for input_ids, labels in tqdm(train_loader, desc='Train'):
        input_ids, labels = input_ids.to(device), labels.to(device)
        logits = model_sas(input_ids)  # (B, L, V)
        B, L, V = logits.shape
        loss = criterion_ce(logits.view(B*L, V), labels.view(B*L))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model_sas.eval()
    # validation loss over all tokens (same as train)
    val_loss = 0
    for input_ids, labels in tqdm(val_seq_loader, desc='Val Loss'):
        input_ids, labels = input_ids.to(device), labels.to(device)
        logits = model_sas(input_ids)
        B, L, V = logits.shape
        val_loss += criterion_ce(logits.view(B*L, V), labels.view(B*L)).item()
    val_loss /= len(val_seq_loader)

    # recommendation metrics on last item
    all_scores, all_labels = [], []
    for inp, slens, tgt in val_rec_loader:
        inp, slens, tgt = inp.to(device), slens.to(device), tgt.to(device)
        logits_full = model_sas(inp)  # (B, L, V)
        B, L, V = logits_full.shape
        for b in range(B):
            last_idx = slens[b] - 1
            scores = logits_full[b, last_idx].clone()
            seen_ids = set(inp[b, -slens[b]:].tolist())
            seen_ids.discard(tgt[b].item())
            for i in seen_ids:
                if i != 0:
                    scores[i] = float('-inf')
            all_scores.append(scores.detach().cpu().numpy())
            all_labels.append(tgt[b].item())

    metrics = evaluate_ranking(np.stack(all_scores), np.array(all_labels))

    wandb.log({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'P@5': metrics['P@5'], 'R@5': metrics['R@5'], 'F1@5': metrics['F1@5'],
        'P@10': metrics['P@10'], 'R@10': metrics['R@10'], 'F1@10': metrics['F1@10']
    })
    print(f"Epoch {epoch:02d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | P@5 {metrics['P@5']:.4f}")

Val Loss: 100%|██████████| 250/250 [00:00<00:00, 360.15it/s]


Epoch 01 | Train: 5.5825 | Val: 5.0540 | P@5 0.0094


Val Loss: 100%|██████████| 250/250 [00:00<00:00, 361.46it/s]


Epoch 02 | Train: 4.9626 | Val: 4.7574 | P@5 0.0042


Val Loss: 100%|██████████| 250/250 [00:00<00:00, 359.33it/s]


...

Epoch 48 | Train: 3.0331 | Val: 2.5868 | P@5 0.0042


Val Loss: 100%|██████████| 250/250 [00:00<00:00, 362.76it/s]


Epoch 49 | Train: 3.0401 | Val: 2.5607 | P@5 0.0048


Val Loss: 100%|██████████| 250/250 [00:00<00:00, 366.48it/s]


Epoch 50 | Train: 3.0157 | Val: 2.5988 | P@5 0.0050


In [ ]:
import random
import torch

# 1. 시퀀스 길이 및 아이템 사전 준비
max_len     = config['max_seq_length']
unique_items = list(token2id.keys())

# 2. 랜덤 시퀀스 생성 (길이 8~12)
random_seqs = []
for _ in range(10):
    length = random.randint(8, 12)
    seq = random.choices(unique_items, k=length)
    random_seqs.append(seq)

# 3. 모델을 평가 모드로 전환
model_sas.eval()
with torch.no_grad():
    for idx, seq in enumerate(random_seqs, 1):
        # 토큰 → ID, truncate, left-pad
        ids    = [token2id[t] for t in seq if t in token2id]
        ids    = ids[-max_len:]
        seqlen = len(ids)
        pad    = [0] * (max_len - seqlen)
        input_ids = torch.tensor([pad + ids],
                                 dtype=torch.long,
                                 device=device)  # (1, L)

        # 모델 통과 → (1, L, V)
        logits_full = model_sas(input_ids)
        last_logits = logits_full[0, seqlen-1]  # (V,)

        # 입력에 존재한 아이템 제외
        for i in set(ids):
            last_logits[i] = float('-inf')

        # Top-5 추천
        topk_ids    = torch.topk(last_logits, k=5).indices.tolist()
        topk_tokens = [id2token[i] for i in topk_ids]

        print(f"\nTest#{idx}  입력시퀀스: {seq}")
        print(f"추천 Top-5 (중복 제거): {topk_tokens}")



Test#1  입력시퀀스: ['ScikitLearn', 'TYPE_Proj|ROLE_DE|SKILL_Numpy', 'Spring', 'TYPE_Proj|ROLE_BE|SKILL_Jenkins', 'TYPE_Junior|ROLE_BE|SKILL_Docker', 'TYPE_Club|ROLE_FE|SKILL_React', 'TYPE_Proj|ROLE_AI|SKILL_PostgreSQL', 'TYPE_StartUp|ROLE_FE|SKILL_Nuxt', 'TYPE_Proj|ROLE_FE|SKILL_Redux', 'TYPE_Hackathon|ROLE_FE|SKILL_VueJS']
추천 Top-5 (중복 제거): ['TYPE_Junior|ROLE_AI|SKILL_Git', 'Python', 'TYPE_StartUp|ROLE_DEVOPS|SKILL_Docker', 'TYPE_StartUp|ROLE_DEVOPS|SKILL_Kubernetes', 'TYPE_Junior|ROLE_AI|SKILL_Pandas']

Test#2  입력시퀀스: ['TYPE_StartUp|ROLE_BE|SKILL_Express', 'TYPE_Junior|ROLE_UXUI|SKILL_NUL', 'TYPE_Proj|ROLE_AI|SKILL_TensorFlow', 'TYPE_Maintenance|ROLE_FE|SKILL_Nuxt', 'TYPE_StartUp|ROLE_AI|SKILL_AWS', 'TYPE_Junior|ROLE_AI|SKILL_SQL', 'TYPE_Proj|ROLE_DE|SKILL_Oracle', 'TYPE_StartUp|ROLE_AI|SKILL_DynamoDB', 'TYPE_StartUp|ROLE_UXUI|SKILL_NUL', 'TYPE_Junior|ROLE_BE|SKILL_Kotlin', 'TYPE_Proj|ROLE_BE|SKILL_Bash']
추천 Top-5 (중복 제거): ['HTMLCSS', 'TS', 'Redux', 'TYPE_Junior|ROLE_GAME|SKILL_NUL', 'R

## SASRec V2

In [1]:
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import wandb
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# 0) 재현성을 위한 시드 고정
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# ─────────────────────────────────────────────────────────────────────────────

# ==================== 1. Load & Preprocess ====================
with open('./util/result_clean.json','r') as f:
    raw = json.load(f)

rows = []
for uid, user in enumerate(raw):
    for ts, tok in enumerate(user['token_sequence']):
        rows.append((uid, tok, ts))
_df = pd.DataFrame(rows, columns=['user_id','item_id','timestamp'])
user_seqs = _df.groupby('user_id')['item_id'].apply(list).tolist()

# 이제 “길이 >= 3인 시퀀스만 사용” (leave-two-out을 적용하려면 최소 3개 이상 필요)
valid_seqs = [seq for seq in user_seqs if len(seq) >= 3]

# 토큰 ↔ ID 매핑
unique_items = sorted(_df['item_id'].unique())
token2id = {t: i+1 for i, t in enumerate(unique_items)}  # 0 = PAD
id2token = {i: t for t, i in token2id.items()}

# ==================== 2. Dataset 정의 (변경 없음) ====================
class SASRecDataset(Dataset):
    """
    Hold-out the last item for recommendation evaluation.
    (validation/test / target 단일 아이템 예측용)
    """
    def __init__(self, sequences, t2i, max_len=20):
        self.seqs    = sequences
        self.t2i     = t2i
        self.max_len = max_len

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        ids = [self.t2i[t] for t in self.seqs[idx] if t in self.t2i]
        ids = ids[-self.max_len:]
        prefix, target = ids[:-1], ids[-1]
        L = len(prefix)
        pad_len = self.max_len - L
        # input_ids: [PAD ... PAD] + [item1, item2, ..., item_L]
        input_ids = [0]*pad_len + prefix
        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(L,         dtype=torch.long),
            torch.tensor(target,    dtype=torch.long)
        )

# ==================== 3. SASRec 모델 정의 (변경 없음) ====================
class SASRec(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.max_len     = cfg['max_seq_length']
        self.hidden_size = cfg['hidden_size']
        self.n_items     = cfg['n_items']

        # item embedding (padding_idx=0)
        self.item_emb = nn.Embedding(self.n_items+1, self.hidden_size, padding_idx=0)
        self.pos_emb  = nn.Embedding(self.max_len, self.hidden_size)

        enc = nn.TransformerEncoderLayer(
            d_model=self.hidden_size,
            nhead=cfg['n_heads'],
            dim_feedforward=cfg['inner_size'],
            dropout=cfg['hidden_dropout_prob'],
            activation=cfg['hidden_act'],
            layer_norm_eps=cfg['layer_norm_eps'],
            batch_first=True
        )
        self.encoder     = nn.TransformerEncoder(enc, num_layers=cfg['n_layers'])
        self.layer_norm  = nn.LayerNorm(self.hidden_size, eps=cfg['layer_norm_eps'])
        self.dropout     = nn.Dropout(cfg['hidden_dropout_prob'])
        self.output_bias = nn.Parameter(torch.zeros(self.n_items+1))

        self._init_weights(cfg['initializer_range'])

    def _init_weights(self, std):
        for n, p in self.named_parameters():
            if 'weight' in n:
                nn.init.normal_(p, mean=0.0, std=std)
            elif 'bias' in n:
                nn.init.constant_(p, 0.0)

    def forward(self, input_ids):
        B, L = input_ids.size()
        # position embedding
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)
        x = self.item_emb(input_ids) + self.pos_emb(pos)
        x = self.layer_norm(x)
        x = self.dropout(x)

        # causal mask (upper triangular)
        mask = torch.triu(torch.ones((L, L), device=x.device), diagonal=1).bool()
        h = self.encoder(x, mask=mask)

        # 최종 logits: (B, L, V)
        logits = torch.matmul(h, self.item_emb.weight.t()) + self.output_bias
        return logits

# ==================== 4. Config & W&B 초기화 ====================
config = {
    'n_layers':2, 'n_heads':2, 'hidden_size':64,
    'inner_size':256, 'hidden_dropout_prob':0.5,
    'hidden_act':'gelu', 'layer_norm_eps':1e-12,
    'initializer_range':0.02,
    'max_seq_length':20,
    'n_items': len(token2id)
}

# ==================== 5. Leave-two-out 분할 및 DataLoaders 설정 ====================
# 5.1) Leave-two-out 방식으로 train_seqs, val_seqs, test_seqs 생성
train_seqs = []
val_seqs   = []
test_seqs  = []

for seq in valid_seqs:
    # 길이가 3 이상이므로, 마지막 두 개를 val/test으로 할당
    # train: seq[:-2], val: seq[:-1], test: seq (원본 전체)
    train_seqs.append(seq[:-2])  # 마지막 두 개 제외
    val_seqs.append(seq[:-1])    # 마지막 하나 제외 (validation 에서는 penultimate 예측)
    test_seqs.append(seq)        # 원본 전체 (test 에서는 마지막 아이템 예측)

# 5.2) Dataset & DataLoader 생성
BATCH_SIZE = 128

# ─ Train DataLoader ─
#   train_seqs마다 “마지막 아이템”을 target으로 삼아 학습
train_rec_ds     = SASRecDataset(train_seqs, token2id, max_len=config['max_seq_length'])
train_rec_loader = DataLoader(train_rec_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)

# ─ Validation DataLoader ─
#   val_seqs마다 SASRecDataset이 마지막 non-pad 위치(=원본 penultimate)만 마스킹 → 그 위치의 target 예측 평가
val_rec_ds     = SASRecDataset(val_seqs,   token2id, max_len=config['max_seq_length'])
val_rec_loader = DataLoader(val_rec_ds,     batch_size=BATCH_SIZE, shuffle=False)

# ─ Test DataLoader ─
#   test_seqs마다 SASRecDataset이 마지막 non-pad 위치(=원본 마지막)만 마스킹 → 그 위치의 target 예측 평가
test_rec_ds     = SASRecDataset(test_seqs,  token2id, max_len=config['max_seq_length'])
test_rec_loader = DataLoader(test_rec_ds,    batch_size=BATCH_SIZE, shuffle=False)

# ==================== 6. Loss, Model, Optimizer ====================
device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_sas    = SASRec(config).to(device)
criterion_ce = nn.CrossEntropyLoss(ignore_index=0)
optimizer    = torch.optim.Adam(model_sas.parameters(), lr=1e-4)

# ==================== 7. 평가 지표 헬퍼 (evaluate_ranking) ====================
def evaluate_ranking(all_scores, all_labels, ks=[1,5,10]):
    N, V = all_scores.shape
    rank = np.argsort(-all_scores, axis=1)
    metrics = {}
    for k in ks:
        topk_indices = rank[:, :k]
        hits = np.array([1 if all_labels[i] in topk_indices[i] else 0 for i in range(N)])
        prec = hits.mean() / k
        rec  = hits.mean()
        hr   = rec
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

        dcg_list = []
        for i in range(N):
            topk_full = rank[i][:k]
            if all_labels[i] in topk_full:
                r = int(np.where(topk_full == all_labels[i])[0][0])
                dcg_i = 1.0 / np.log2(r + 2)
            else:
                dcg_i = 0.0
            dcg_list.append(dcg_i)
        ndcg = float(np.mean(dcg_list))

        metrics.update({
            f"P@{k}":   prec,
            f"R@{k}":   rec,
            f"HR@{k}":  hr,
            f"F1@{k}":  f1,
            f"nDCG@{k}": ndcg,
        })
    return metrics

# ==================== 8. Train + Validation (Last‐item loss & Rec Metrics) ====================
best_val_loss = float('inf')
best_epoch    = -1

EPOCHS = 100

for epoch in tqdm(range(1, 1 + EPOCHS), desc='Training'):
    # ---------- (1) Train (penultimate이 아니라 “train_seqs의 마지막” 예측) ----------
    model_sas.train()
    train_loss = 0.0
    for inp, slens, tgt in train_rec_loader:
        # inp: (B, L), slens: (B,), tgt: (B,)
        inp, slens, tgt = inp.to(device), slens.to(device), tgt.to(device)

        logits_full = model_sas(inp)  # (B, L, V)

        # ── train_seqs에서는 “train_seqs[i]의 마지막 아이템”을 예측해야 하므로,
        #     slens[b] - 1 위치의 logit만 사용
        B, L, V = logits_full.size()
        logits_last = torch.zeros((B, V), device=inp.device)
        for b in range(B):
            last_idx = slens[b].item() - 1
            logits_last[b] = logits_full[b, last_idx]
        # ─────────────────────────────────────────────────────────────────────────

        loss = criterion_ce(logits_last, tgt)  # (B, V) vs (B,)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_rec_loader)

    # ---------- (2) Validation Loss (penultimate 예측) ----------
    model_sas.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inp, slens, tgt in val_rec_loader:
            inp, slens, tgt = inp.to(device), slens.to(device), tgt.to(device)
            logits_full = model_sas(inp)  # (B, L, V)

            B, L, V = logits_full.size()
            logits_last = torch.zeros((B, V), device=inp.device)
            for b in range(B):
                last_idx = slens[b].item() - 1
                logits_last[b] = logits_full[b, last_idx]

            val_loss += criterion_ce(logits_last, tgt).item()
    val_loss /= len(val_rec_loader)

    # ---------- (3) Validation Rec Metrics (penultimate 예측, 순위 평가) ----------
    all_scores, all_labels = [], []
    with torch.no_grad():
        for inp, slens, tgt in val_rec_loader:
            inp, slens, tgt = inp.to(device), slens.to(device), tgt.to(device)
            logits_full = model_sas(inp)  # (B, L, V)
            B, L, V     = logits_full.shape

            for b in range(B):
                last_idx = slens[b].item() - 1
                scores = logits_full[b, last_idx].clone()  # (V,)

                # 이미 본 아이템들(= prefix)에 대해 score를 -inf 처리
                # prefix 길이는 slens[b], prefix 아이템 ID는 inp[b, -slens[b]:]
                seen_ids = set(inp[b, -slens[b]:].tolist())
                seen_ids.discard(tgt[b].item())
                for i in seen_ids:
                    if i != 0:
                        scores[i] = float('-inf')

                all_scores.append(scores.detach().cpu().numpy())
                all_labels.append(tgt[b].item())

    val_metrics = evaluate_ranking(np.stack(all_scores), np.array(all_labels))

    # ---------- (4) 최적 모델 저장 ----------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch    = epoch
        torch.save(model_sas.state_dict(), "best_sasrec_leave2out.pt")

    # ── 로그 출력 ───────────────────────────────────────────────────────────────────
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

print(f"▶ Best Epoch: {best_epoch} | Val Loss: {best_val_loss:.4f}")

# ==================== 9. Test 평가: 저장된 최적 모델 로드 후 “마지막 아이템” 예측 ====================
# (1) 최적화된 모델 로드
model_sas.load_state_dict(torch.load("best_sasrec_leave2out.pt"))
model_sas.eval()

# (2) Test Loss (마지막 아이템 예측)
test_loss = 0.0
with torch.no_grad():
    for inp, slens, tgt in tqdm(test_rec_loader, desc='Test Loss (Last‐item)'):
        inp, slens, tgt = inp.to(device), slens.to(device), tgt.to(device)
        logits_full = model_sas(inp)  # (B, L, V)

        B, L, V = logits_full.size()
        logits_last = torch.zeros((B, V), device=inp.device)
        for b in range(B):
            last_idx = slens[b].item() - 1
            logits_last[b] = logits_full[b, last_idx]

        test_loss += criterion_ce(logits_last, tgt).item()
test_loss /= len(test_rec_loader)

# (3) Test Rec Metrics (hold-out last item, ranking 평가)
all_test_scores, all_test_labels = [], []
with torch.no_grad():
    for inp, slens, tgt in test_rec_loader:
        inp, slens, tgt = inp.to(device), slens.to(device), tgt.to(device)
        logits_full = model_sas(inp)
        B, L, V     = logits_full.shape

        for b in range(B):
            last_idx = slens[b].item() - 1
            scores = logits_full[b, last_idx].clone()

            # 이미 본 아이템(= penultimate 포함)들에 대해 score -inf 처리
            seen_ids = set(inp[b, -slens[b]:].tolist())
            seen_ids.discard(tgt[b].item())
            for i in seen_ids:
                if i != 0:
                    scores[i] = float('-inf')

            all_test_scores.append(scores.detach().cpu().numpy())
            all_test_labels.append(tgt[b].item())

test_metrics = evaluate_ranking(np.stack(all_test_scores), np.array(all_test_labels))

# ── 결과 출력 ───────────────────────────────────────────────────────────────────
print("\n===== Test 결과 (Last‐item 기준) =====")
print(f"Test Loss: {test_loss:.4f}")
print(f"HR@1   : {test_metrics['HR@1']:.4f}")
print(f"HR@5   : {test_metrics['HR@5']:.4f}")
print(f"HR@10  : {test_metrics['HR@10']:.4f}")
print(f"NDCG@5 : {test_metrics['nDCG@5']:.4f}")
print(f"NDCG@10: {test_metrics['nDCG@10']:.4f}")
print(f"MRR    : {test_metrics['P@1']:.4f}")


Training:   1%|          | 1/100 [00:04<08:06,  4.91s/it]

Epoch 01 | Train Loss: 5.2184 | Val Loss: 5.2117


Training:   2%|▏         | 2/100 [00:09<07:57,  4.87s/it]

Epoch 02 | Train Loss: 5.1965 | Val Loss: 5.1890


Training:   3%|▎         | 3/100 [00:14<07:50,  4.85s/it]

...

Epoch 99 | Train Loss: 3.5742 | Val Loss: 4.1184


Training: 100%|██████████| 100/100 [08:00<00:00,  4.81s/it]


Epoch 100 | Train Loss: 3.5757 | Val Loss: 4.1343
▶ Best Epoch: 88 | Val Loss: 4.1051


Test Loss (Last‐item): 100%|██████████| 47/47 [00:00<00:00, 108.93it/s]



===== Test 결과 (Last‐item 기준) =====
Test Loss: 4.3962
HR@1   : 0.0547
HR@5   : 0.2630
HR@10  : 0.4028
NDCG@5 : 0.1601
NDCG@10: 0.2045
MRR    : 0.0547
